# Notebook 27 — Generation, Batching, Streaming, and Structured Output

    ## Learning objectives

    - Choose decoding parameters based on task requirements
- Batch variable-length prompts and separate prompt/output tokens
- Stream and validate schema-constrained responses

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'huggingface-hub>=0.30,<1', 'python-dotenv>=1.1', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 27.1 Decoding is part of the product contract

Greedy decoding is reproducible but not universally best. Temperature rescales logits;
top-k keeps k candidates; top-p keeps the smallest set reaching cumulative probability
p. Repetition penalties modify likelihoods. Seeds improve repeatability but do not
guarantee identical results across hardware or software versions. Evaluate the complete
model + prompt + decoding configuration.


In [ ]:
import torch
logits = torch.tensor([3.0, 2.0, 1.0, 0.0])
for temperature in [0.5, 1.0, 2.0]:
    print(temperature, torch.softmax(logits / temperature, -1).numpy().round(3))


## 27.2 Batch locally

Decoder-only models should generally left-pad for batched generation so the final
non-padding position aligns across examples. Slice each output after the padded input
width, not after the original text length. Batch throughput can rise while per-request
latency worsens; measure both.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
device = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
lm = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto").to(device)
prompts = ["The capital of France is", "In one phrase, gradient accumulation"]
batch = tok(prompts, padding=True, return_tensors="pt").to(lm.device)
outputs = lm.generate(**batch, max_new_tokens=24, do_sample=False)
generated = outputs[:, batch["input_ids"].shape[1]:]
print(tok.batch_decode(generated, skip_special_tokens=True))


In [ ]:
# Optional remote structured output.
import json, os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")
if token:
    client = InferenceClient(token=token)
    schema = {"type": "json_schema", "json_schema": {"name": "concept",
        "schema": {"type": "object", "properties": {
            "term": {"type": "string"}, "definition": {"type": "string"}},
            "required": ["term", "definition"], "additionalProperties": False},
        "strict": True}}
    try:
        result = client.chat_completion(model=os.getenv("HF_CHAT_MODEL", "Qwen/Qwen2.5-7B-Instruct-1M"),
            messages=[{"role": "user", "content": "Define perplexity."}],
            response_format=schema, max_tokens=120, temperature=0)
        print(json.loads(result.choices[0].message.content))
    except Exception as exc:
        print("Provider/model may not support this schema:", type(exc).__name__)
else:
    print("Remote cell skipped: HUGGINGFACE_TOKEN is not configured.")


## 27.3 Streaming semantics

Streaming improves perceived latency but not necessarily total latency. Clients must
handle partial UTF-8/text, finish reasons, usage arriving at the end, cancellation,
disconnects, and moderation decisions. Never parse incomplete JSON as final output.


## 27.4 Sampling algorithms step by step

Repetition penalties modify logits before filtering. Temperature rescales. Top-k masks all but
k highest logits. Top-p sorts probabilities and retains the smallest prefix whose cumulative
mass crosses p. Min-p keeps tokens relative to the best probability. Typical sampling favors
tokens near expected information content. The order and exact implementation are library
behavior; inspect `GenerationConfig` for the installed version.

Greedy decoding is not the same as globally most likely sequence: it selects the best token at
each step. Beam search tracks multiple sequence hypotheses and is useful for constrained
sequence tasks, but can produce generic text for open-ended chat. Sampling gives a distribution
of outcomes. For evaluation, fix seeds and repeat stochastic calls; for production, choose
parameters based on measured task success, diversity, safety, latency, and output length.


In [ ]:
# Implement top-k/top-p filtering on one logit vector.
def filter_logits(logits, top_k=None, top_p=None):
    values = logits.clone()
    if top_k:
        threshold = torch.topk(values, min(top_k, values.numel())).values[-1]
        values[values < threshold] = -torch.inf
    if top_p is not None:
        sorted_logits, indices = torch.sort(values, descending=True)
        probs = sorted_logits.softmax(-1)
        remove = probs.cumsum(-1) - probs > top_p
        values[indices[remove]] = -torch.inf
    return values

demo = torch.tensor([4., 3., 2., 1., 0.])
for kwargs in [{}, {"top_k": 2}, {"top_p": .8}, {"top_k": 4, "top_p": .8}]:
    filtered = filter_logits(demo, **kwargs)
    print(kwargs, filtered.softmax(-1))


## 27.5 Stopping, length, and reproducibility

`max_new_tokens` caps generated tokens and is generally clearer than total `max_length`.
EOS lets the model stop naturally; custom stop strings require token-aware or incremental
matching and may span token boundaries. A finish reason of length means the result may be
incomplete. Forced minimum lengths can encourage filler. For structured output, reserve enough
tokens to close the structure and still validate it.

Seeds control random number streams but full determinism also depends on hardware, kernels,
batching, precision, library versions, and server behavior. Dynamic batching can alter numeric
paths. Store raw outputs and complete generation config. For regression tests, greedy decoding
is convenient but may not represent the production sampling distribution; maintain both a
deterministic suite and repeated stochastic quality estimates.


In [ ]:
# Compare deterministic and sampled continuations using the already loaded small model.
prompt = tok("A reliable evaluation should", return_tensors="pt").to(lm.device)
configs = [
    {"do_sample": False},
    {"do_sample": True, "temperature": .7, "top_p": .9},
    {"do_sample": True, "temperature": 1.2, "top_k": 20},
]
for i, config in enumerate(configs):
    torch.manual_seed(123)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(123)
    result = lm.generate(**prompt, max_new_tokens=30, **config)
    print(i, tok.decode(result[0, prompt["input_ids"].shape[1]:], skip_special_tokens=True))


## 27.6 Batching and streaming implementation reference

Throughput batching groups prompts into one forward pass. Padding waste grows with length
variance, so bucket by length. Batch size is constrained by prompt tokens, expected output,
KV cache, and allocator workspace—not examples alone. One long generation keeps the batch
active unless the engine supports continuous batching. Local `generate()` is not a substitute
for a concurrent inference server.

Streaming transports deltas, not independent complete messages. Accumulate by choice/index;
handle role/tool deltas, empty chunks, usage/final metadata, finish reasons, disconnects, and
cancellation. Rendering untrusted HTML/Markdown incrementally introduces security concerns.
Measure queue time and TTFT separately from generation time. A canceled client should cancel
upstream compute where supported rather than merely stop displaying tokens.

Structured generation constrains syntax according to JSON Schema/grammar, but semantic
validation remains application work. Parse once complete, validate types/ranges/enums and
business rules, reject extra properties, and define retries. Record provider fallback from
strict schema to JSON mode because it changes reliability.


## 27.7 Generation reference

| Parameter | Effect | Common misuse |
|---|---|---|
| `max_new_tokens` | Hard output cap | Confusing with total context length |
| `temperature` | Rescales logits | Treating zero as universal quality setting |
| `top_k` | Keeps k candidates | Comparing k across very different distributions |
| `top_p` | Keeps cumulative mass | Assuming it guarantees factuality |
| repetition penalty | Modifies seen-token logits | Damaging code/names or required repetition |
| stop/EOS | Terminates generation | Matching strings without token-boundary care |
| seed/generator | Controls RNG stream | Claiming cross-system bitwise determinism |

Record prompt after template, model/revision, tokenizer, generation config, seed, finish reason, and
raw IDs/output. Slice generated IDs after padded input width. In batches, configure padding side and
pad token deliberately. For sampling comparisons, use repeated paired prompts and include output
length because decoding settings change how long the model speaks.

Structured output guarantees at most syntactic/schema compliance supported by the engine. Validate
business semantics and unknown fields, define retry/repair policy, and treat parser failure as an
observed outcome. Streaming must accumulate before final structured parsing.


## 27.8 Decode only the continuation

Batched causal generation returns prompt plus continuation and prompts can have different padded lengths. Slice with the encoded input width used by `generate`, then use attention-mask lengths only when mapping logical prompt positions. Left padding is often preferred for decoder-only batched generation, but model support and position handling must be checked. Do not strip strings by prefix matching because normalization and template tokens make it unreliable. Stop conditions should be token-aware and per sequence.


In [ ]:
input_ids=torch.tensor([[0,0,5,6],[0,7,8,9]]); attention=input_ids.ne(0); generated=torch.tensor([[0,0,5,6,10,11],[0,7,8,9,12,2]])
continuations=generated[:,input_ids.shape[1]:]; print(attention.sum(-1),continuations)


## 27.9 Structured generation has two layers

Constrained decoding restricts which tokens can be emitted; validation checks the parsed semantic object. Both are necessary. A string may satisfy a JSON grammar while violating required fields, ranges, cross-field relationships, or authorization. Define a schema, validate after generation, return typed errors, and decide whether bounded repair is safe. Measure first-pass validity separately from post-repair validity and correctness. Never execute model-produced code or tool arguments merely because they parse.


In [ ]:
def validate_order(x):
 required={"item","quantity"}
 if set(x)!=required: return False,"fields"
 if not isinstance(x["quantity"],int) or not 1<=x["quantity"]<=10: return False,"quantity"
 return True,"ok"
for obj in [{"item":"book","quantity":2},{"item":"book","quantity":99},{"item":"book","quantity":"2"}]: print(obj,validate_order(obj))


## Exercises

    1. Plot output diversity and task success across three temperatures.
2. Benchmark batch sizes 1, 2, 4, and 8 with fixed token lengths.
3. Implement a streaming accumulator that records TTFT and final usage.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
